# 01 — Exploratory Data Analysis

**Goal:** understand the CICIDS2017 dataset before touching any model.

This notebook does **not** re-implement EDA. It calls into `src.data.eda` so the code lives in one place and tests can cover it.

If you don't have CICIDS2017 downloaded yet, run this first to generate a synthetic stand-in:

```powershell
python scripts/generate_sample.py
```

In [ ]:
# Imports and logging setup. Run this cell first.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.logging import configure_logging
configure_logging(level='INFO')

from src.config.constants import (
    SAMPLE_DIR, RAW_DIR, TARGET_LABELS, LABEL_COLUMN,
)

## Step 1 — Load data

By default this notebook reads from `data/sample/` (synthetic data) so you can run it without the real CSVs. Once you've downloaded CICIDS2017 to `data/raw/`, change `data_dir` below to `RAW_DIR`.

In [ ]:
from src.data.loader import load_raw

data_dir = SAMPLE_DIR  # change to RAW_DIR once CICIDS2017 is in data/raw/
df = load_raw(raw_dir=data_dir)
print('Loaded shape:', df.shape)
df.head()

## Step 2 — Clean

CICIDS ships with three known data-quality issues:

1. **Leading whitespace** in every column name.
2. **±Inf** values in speed columns (Flow Bytes/s, Flow Packets/s).
3. **Duplicate rows** in the Friday CSVs.

`src.features.cleaning.clean` handles all three. We then filter to the 4 target classes.

In [ ]:
from src.features.cleaning import clean, filter_target_classes

df_clean = clean(df)
df_clean = filter_target_classes(df_clean, TARGET_LABELS)
print('After cleaning + filtering:', df_clean.shape)
df_clean[LABEL_COLUMN].value_counts()

## Step 3 — Summary statistics

`describe_dataset` returns a JSON-serializable dict so the same code feeds the dashboard and reports.

In [ ]:
from src.data.eda import describe_dataset

summary = describe_dataset(df_clean)
summary

## Step 4 — Plots

`run_eda` saves four PNGs under `results/figures/`. The dashboard's EDA page picks them up from there.

In [ ]:
from src.data.eda import run_eda

summary = run_eda(df_clean)
for name, path in summary['figures'].items():
    print(f'{name:25s} -> {path}')

## What we learned

Fill in here after running, e.g.:

- Class distribution is heavily skewed toward BENIGN (~55%), DoS Hulk is the most common attack class.
- N features have ±Inf in the raw data; cleaning drops them.
- Correlation heatmap shows clusters around packet-length and IAT features — strong candidates for feature selection in Phase 5.